# SFT Stage-7 — run on the A100 (open THIS notebook in Cursor)

Connect the kernel to your **A100** (kernel selector → Colab → **Auto Connect**), then run top to bottom. **CELL 0 self-provisions** the runtime (clone/install/token/stage), so a fresh or recycled VM works — no separate readiness notebook needed.

The greedy improve goal (pasted into Codex) rewrites the tagged candidate cell and Computer Use presses **Run All**; each `METRIC=` line lands in this local file for Codex to read.

## CELL 0 — provision the runtime (idempotent; run first)
Works on a **fresh OR warm** A100. Clones the repo, installs deps, loads `HF_TOKEN`, and stages the corpus **only if missing**. ⚠ On a brand-new VM this does the ~9.85 GB staging pull (a few minutes). If your earlier runtime is still alive, prefer reconnecting to it (**Auto Connect**) so this is a no-op.

In [ ]:
import os, pathlib, subprocess, sys

REPO = '/content/SLM'
# 1) clone if missing (idempotent), then cd in
if not pathlib.Path(REPO, '.git').is_dir():
    print('cloning -> /content/SLM ...')
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)

# 2) checkout + pull + install ONCE per session (so a Run-All loop does not re-pull / re-install
#    or change code mid-run). SLM_PROVISIONED makes re-runs a fast no-op.
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'checkout', 'main'], check=True)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

# 3) HF token for the PRIVATE dataset + Qwen download. Prefer an uploaded .env, else paste (read scope OK).
def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
if not os.environ.get('HF_TOKEN'):
    tok = _env_token('HF_TOKEN')
    if not tok:
        import getpass
        tok = getpass.getpass('Paste HF_TOKEN (read scope OK for staging): ').strip()
    os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

# 4) stage the corpus ONLY if missing (weights are gitignored -> come only from staging; ~9.85GB)
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')
print('contents:', sorted(os.listdir('/content/slm')) if pathlib.Path('/content/slm').is_dir() else 'MISSING')


In [ ]:
import os, getpass, pathlib

_env_path = pathlib.Path('/content/.env')
_saved = {}
if _env_path.is_file():
    for _line in _env_path.read_text().splitlines():
        if '=' in _line and not _line.lstrip().startswith('#'):
            _key, _value = _line.split('=', 1)
            _saved[_key.strip()] = _value.strip().strip('\"').strip("'")
for _name, _prompt in (
    ('HF_TOKEN', 'HF_TOKEN (read ok for staging): '),
    ('HF_WRITE_TOKEN', 'HF_WRITE_TOKEN (SLM_Alpha_Write): '),
):
    if not os.environ.get(_name):
        os.environ[_name] = _saved.get(_name) or getpass.getpass(_prompt).strip()
if not os.environ.get('HF_TOKEN') or not os.environ.get('HF_WRITE_TOKEN'):
    raise RuntimeError('HF_TOKEN and HF_WRITE_TOKEN must both be non-empty')
if not _saved.get('HF_TOKEN') or not _saved.get('HF_WRITE_TOKEN'):
    _env_path.write_text(
        f"HF_TOKEN={os.environ['HF_TOKEN']}\nHF_WRITE_TOKEN={os.environ['HF_WRITE_TOKEN']}\n")
print('tokens loaded; read:', bool(os.environ.get('HF_TOKEN')),
      '| write:', bool(os.environ.get('HF_WRITE_TOKEN')), '| env file:', _env_path.is_file())

In [ ]:
import torch, transformers
import bitsandbytes as bnb, peft, accelerate, qwen_vl_utils  # must all import
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor  # VL class must exist
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, 'bnb', bnb.__version__,
      'peft', peft.__version__, 'accelerate', accelerate.__version__)
assert torch.cuda.is_available(), 'no CUDA — wrong runtime (need A100)'
print('imports OK')


In [ ]:
import os, pathlib, shutil
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'   # lowercase corpus (images + tokenizer)
repo, corpus = pathlib.Path('/content/SLM'), pathlib.Path('/content/slm')
assert repo.is_dir(), 'repo clone missing at /content/SLM'
assert corpus.is_dir(), 'staged corpus missing at /content/slm (run CELL 0)'
tok = corpus / 'tokenizer' / 'final' / 'model.pt'
assert tok.is_file(), f'frozen tokenizer weights missing: {tok}'
free_gb = shutil.disk_usage('/content').free / 1e9
# need the big headroom only before the ~12-14GB fp32 base_resized is written; afterwards a small
# margin is enough (avoids a false abort late in a Run-All loop as adapters accumulate).
need_gb = 25 if not pathlib.Path('models/base_resized/vocab_resize_manifest.json').is_file() else 5
print('SLM_ARTIFACT_ROOT =', os.environ['SLM_ARTIFACT_ROOT'], '| free /content GB:', round(free_gb, 1), '| need >', need_gb)
assert free_gb > need_gb, f'need >{need_gb}GB free on /content (have {free_gb:.1f})'
print('pre-run asserts OK')


## CELL 3 — vocab resize + preflight  🛑 expect `[vocab-resize][OK]` + identity bound
Downloads Qwen2.5-VL-3B (~6 GB) and writes a full fp32 `models/base_resized` (~12-14 GB). Run it **once** (unseeded new-token init — re-running trains a different base).

In [ ]:
import pathlib, os, subprocess, sys
# preflight dry-run ONLY if base_resized is not built yet (else it re-downloads Qwen each Run All).
if pathlib.Path('models/base_resized/vocab_resize_manifest.json').is_file():
    print('base_resized exists -> skipping preflight-only dry run.')
else:
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--preflight-only'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})


In [ ]:
import json, pathlib, os, subprocess, sys
D = pathlib.Path('models/base_resized')
# Build base_resized ONCE. Re-running would re-download + re-resize with UNSEEDED new-token init
# (a different base every time), breaking cross-run metric comparability. Skip if already built.
if (D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file():
    print('base_resized already built -> skipping vocab_resize (delete the dir to rebuild).')
else:
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
# identity assert (cheap; runs every pass)
assert (D / 'preprocessor_config.json').is_file(), 'base_resized missing preprocessor_config.json (AutoProcessor would fail)'
m = json.load(open(D / 'vocab_resize_manifest.json'))
EXPECT = 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f'
assert m.get('tokenizer_version') == EXPECT, ('tokenizer_version mismatch/null', m.get('tokenizer_version'))
assert m.get('vq_codebook_sha256'), 'null vq_codebook_sha256 — identity did not bind (check SLM_ARTIFACT_ROOT)'
print('vocab_resize OK; identity bound:', m['tokenizer_version'])


## GATE 0 — one OBSERVABLE smoke run (must pass before the loop)  🛑
Uses `gradient_accumulation_steps=4`/`effective_batch_size=4` so ≥10 optimizer steps land past warmup and loss prints. Confirm: `steps>0`, ≥2 falling loss lines, `[sft][OK]`, and a finite `METRIC=` from the scorer. If `[sft][skip]` approaches the row count → images aren't resolving (case trap) → STOP.

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/sft_default.yaml'))
cfg.update(gradient_accumulation_steps=4, effective_batch_size=4)  # observability on smoke-50
yaml.safe_dump(cfg, open('gate0.yaml','w'))
print('gate0.yaml triple:', {k: cfg[k] for k in ('per_device_batch_size','gradient_accumulation_steps','effective_batch_size')})


In [ ]:
import pathlib, os, subprocess, sys
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
adapter = 'models/sft_adapters/gate0_smoke50'
# Gate 0 is a one-time go/no-go. Skip on re-run so a Codex Run-All loop does not re-train it.
if pathlib.Path(adapter, 'adapter_manifest.json').is_file():
    print('Gate 0 already ran -> skipping (delete', adapter, 'to repeat).')
else:
    subprocess.run([sys.executable, '-m', 'sft.train', '--config', 'gate0.yaml',
                    '--resized-model', 'models/base_resized', '--smoke-size', '50',
                    '--run-id', 'gate0'], check=True, env=env)
    subprocess.run([sys.executable, '-m', 'sft.score_tokens', '--config', 'gate0.yaml',
                    '--resized-model', 'models/base_resized', '--adapter', adapter,
                    '--limit', '32'], check=True, env=env)


## IMPROVE LOOP — candidate eval (driven by Codex)
Codex rewrites the tagged candidate cell below; Computer Use runs **Run All**; the bridge trains + scores and runs a FULL 2-epoch training over the whole corpus (long) and prints one `METRIC=` line that Codex reads from this file.

In [ ]:
# HF UPLOAD SETUP — run once. Adapters are uploaded to PUSH_HF_REPO after each eval.
# Leave PUSH_HF_REPO empty to skip upload. Upload needs a WRITE token (read-only HF_TOKEN 403s).
import os, getpass, pathlib
os.environ['PUSH_HF_REPO'] = 'ericrcwu/LUT_SLM_sft_adapters'   # <-- your HF model repo (created if missing)
if not os.environ.get('HF_WRITE_TOKEN'):
    for _p in ('/content/SLM/.env', '/content/.env'):
        if pathlib.Path(_p).exists():
            for _l in pathlib.Path(_p).read_text().splitlines():
                if _l.strip().startswith('HF_WRITE_TOKEN='):
                    os.environ['HF_WRITE_TOKEN'] = _l.split('=',1)[1].strip().strip('"').strip("'")
if os.environ['PUSH_HF_REPO'] and not os.environ.get('HF_WRITE_TOKEN'):
    os.environ['HF_WRITE_TOKEN'] = getpass.getpass('Paste HF WRITE token (SLM_Alpha_Write): ').strip()
print('PUSH_HF_REPO =', os.environ.get('PUSH_HF_REPO'), '| write token set:', bool(os.environ.get('HF_WRITE_TOKEN')))


In [ ]:
# @candidate-config  (auto-written by the improve loop; do not hand-edit)
import json, pathlib
_CANDIDATE = '{\n  "learning_rate_lora": 0.0003,\n  "lora_r": 16,\n  "lora_alpha": 32,\n  "lora_dropout": 0.05,\n  "warmup_ratio": 0.03,\n  "max_grad_norm": 1.0,\n  "weight_decay": 0.0,\n  "max_pixels": 200704\n}'
pathlib.Path("/content/SLM/candidate.json").write_text(_CANDIDATE)
print("candidate.json written:", _CANDIDATE)


In [ ]:
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.bilevel_bridge --mode colab \
    --config /content/SLM/candidate.json --resized-model models/base_resized \
    --smoke-size 0 --score-limit 128 --timeout 21600 --run-id bl
